# Compare/Find Similar Models: Artificial Analysis Data
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elaheJ/IT344-IllinoisStateUniversity-Model-Comparison-using-Artificial-Analysis-Utility-Colabs/blob/main/Compare_Find_Similar_Models_Artificial_Analysis_Data.ipynb)

**IT344 · Illinois State University**

Choose a model for your application by comparing quality, cost, and response time. Find alternatives similar to a model you already use.

**Opening this notebook does not start the app. Run the single cell below to display its controls.**

1. In **Secrets** (🔑 in the left sidebar), add **AA_API_KEY** and enable **Notebook access**. Keep your key out of code cells.
2. A standard CPU runtime is enough.
3. Click **▶ Start the app** below, or choose **Runtime → Run all**. Wait for the three setup stages to finish. The Gradio app appears **directly below that cell**.

**Choices in the app:** select models, set project requirements, choose one metric or a weighted composite, or open **Find similar models** and choose a reference model and features. All calculations in this notebook use Python; no LLM is needed.

If the embedded view is blank, open the **gradio.live** link printed in the output. Use **File → Save a copy in Drive** if you want to keep your own notebook.


In [ ]:
# @title ▶ Start the app
# Install dependencies and load the small shared classroom source package.
import io
import os
from pathlib import Path
import subprocess
import ssl
import sys
import tempfile
import urllib.request
import zipfile

os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
packages = ['gradio==6.0.2', 'pandas>=2.2,<3', 'plotly>=5.24,<7', 'certifi>=2024.2.2', 'huggingface-hub>=0.34,<1']
print("1/3 · Installing app dependencies. Please wait…", flush=True)
installed = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], capture_output=True, text=True)
if installed.returncode:
    print(installed.stderr[-3000:])
    raise RuntimeError("Installation failed. Reconnect to a fresh Colab runtime and rerun this cell.")

# Load a fixed, published version of the shared app code.
# AA data comes from your own Colab Secret below.
source_dir = Path(tempfile.mkdtemp(prefix="it344-colab-"))
source_url = "https://codeload.github.com/elaheJ/ai-model-comparison-using-artificial-analysis-dev/zip/5a0b1085b9f14c12f475f16c9b27744de219417c"
needed = ["__init__.py", "core.py", "snapshots.py", "agent.py", "local_qwen.py", "colab_ui.py"]
import certifi
context = ssl.create_default_context()
context.load_verify_locations(cafile=certifi.where())
with urllib.request.urlopen(source_url, timeout=60, context=context) as response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
with archive:
    archive_root = archive.namelist()[0].split("/")[0]
    for filename in needed:
        member = archive_root + "/model_compare/" + filename
        target = source_dir / "model_compare" / filename
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))
sys.path.insert(0, str(source_dir))

from IPython.display import HTML, display
import gradio as gr
from model_compare.colab_ui import CSS, build_app, load_colab_snapshot

if "app" in globals():
    app.close()
print("2/3 · Reading AA_API_KEY from Colab Secrets and fetching model data…", flush=True)
snapshot = None
snapshot = load_colab_snapshot()
if snapshot is None:
    display(HTML("<p><b>The app has not started.</b> Fix the Secrets or data-loading issue shown above, then run this cell again.</p>"))
else:
    print("3/3 · Starting the app. The controls will appear below…", flush=True)
    app = build_app(snapshot, agent=False)
    app.queue(default_concurrency_limit=1, max_size=8)
    theme = gr.themes.Base(primary_hue="red", neutral_hue="gray").set(
        button_primary_background_fill="#cc0000",
        button_primary_background_fill_hover="#a30000",
        button_primary_text_color="#ffffff", body_background_fill="#ffffff")
    app.launch(share=True, inline=True, height=1000, show_error=False, css=CSS, theme=theme)
    if app.share_url:
        display(HTML(f'<p><a href="{app.share_url}" target="_blank" rel="noopener"><b>Open the full app in a new tab ↗</b></a></p>'))


### Use the results for your project
Build a shortlist, test it on real application examples, and justify your choice with evidence. Download the CSV or JSON results before disconnecting. Similarity measures closeness, not superiority; missing measurements are not zero.

**Credit:** Benchmark and pricing data come from [Artificial Analysis](https://artificialanalysis.ai/). Each run fetches fresh data using your Colab Secret. 

Keep the runtime connected while using the app. Its temporary Gradio link can be opened by anyone who has it; the key is never passed to the app. If a network filter blocks that link, ask your network administrator to review it.

[Utility notebooks](https://github.com/elaheJ/IT344-IllinoisStateUniversity-Model-Comparison-using-Artificial-Analysis-Utility-Colabs) · [Shared app source](https://github.com/elaheJ/ai-model-comparison-using-artificial-analysis-dev/tree/5a0b1085b9f14c12f475f16c9b27744de219417c)
